# 02 — Préparation & feature engineering multi-tables

**Objectif :** produire une table **1 ligne = 1 `SK_ID_CURR`**, enrichie par agrégation des tables satellites, sans leakage.

**Entrées :** `data/raw/*.csv`  
**Sorties :** `data/interim/{train,test}_full.parquet`, `data/processed/{train,valid,test}.parquet`, `data/processed/meta.json`

**Implémentation retenue (`fe_v1`) :** `src/features.py` + `src/build_dataset.py` (journal dans `choix_techniques.txt`).

**Hors scope de ce notebook :** entraînement de modèles, MLflow, seuil métier, SHAP.

---

### Règles anti-leakage appliquées
1. Agrégation des tables satellites **avant** le merge (pas d’explosion 1→N dans la table finale).
2. Split train/valid **stratifié** avant tout *fit* d’imputation / encoding / scaling (ces fits sont dans les pipelines modèles).
3. Même logique FE sur `application_test` ; **aucun** fit qui utilise le test ni le valid.
4. Pas d’information postérieure à la demande : les `DAYS_*` Home Credit sont déjà relatifs à la date de demande.

### Suite de l’EDA (faits repris dans `fe_v1`)
- Déséquilibre ~8 % → split stratifié (`test_size=0.2`, `random_state=42`).
- Bloc logement très NA → flag `HOUSING_MISSING`.
- `EXT_SOURCE_*` conservés.
- Sentinelle `DAYS_EMPLOYED = 365243` → NaN + `DAYS_EMPLOYED_ANOMALY` ; dérivés `AGE_YEARS`, `EMPLOYMENT_YEARS`, ratios crédit/revenu.


## 0. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

# TODO : imports preprocessing (sklearn, etc.) quand tu en auras besoin

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("RAW =", RAW)
assert (RAW / "application_train.csv").exists()

## 1. Chargement des tables

Les CSV Home Credit sont chargés depuis `data/raw/`. Les tables les plus volumineuses sont `bureau_balance` et `installments_payments`. L’option `--no-bb` de `build_dataset` permet d’exclure `bureau_balance` pour un run plus rapide.


In [ ]:
# TODO : charger application_train / application_test
# TODO : charger les tables satellites dont tu as besoin
# Astuce : commencer par application + bureau + previous_application,
#          puis ajouter POS / CC / installments / bureau_balance si pertinent.

train = None  # pd.read_csv(...)
test = None

# bureau = ...
# previous = ...
# ...

## 2. Nettoyage minimal sur `application_*`

Traitements retenus en `fe_v1` (transformations déterministes, sans fit d’échantillon) :
- sentinelle `DAYS_EMPLOYED == 365243` → NaN + indicateur ;
- `AGE_YEARS`, `EMPLOYMENT_YEARS` ;
- ratios `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_GOODS_RATIO` ;
- flag `HOUSING_MISSING` pour le bloc logement.

L’imputation, le one-hot encoding et le scaling (régression logistique / random forest) sont fit **après** le split, dans `src/modeling.py`.


In [ ]:
def clean_application(df: pd.DataFrame) -> pd.DataFrame:
    """Transformations déterministes (pas de fit sur données)."""
    df = df.copy()
    # TODO : sentinelle DAYS_EMPLOYED
    # TODO : features simples (âge, ratios, flags NA...)
    return df

# train = clean_application(train)
# test = clean_application(test)

## 3. Agrégations des tables satellites

Chaque table 1→N est réduite à **une ligne par `SK_ID_CURR`** avant merge.

| Table | Grain initial | Clé d’agrégation | Features produites (exemples `fe_v1`) |
|-------|---------------|------------------|----------------------------------------|
| `bureau` | 1 crédit bureau | `SK_ID_CURR` | counts, sommes / moyennes de dettes, retards, prolongations, répartition `CREDIT_ACTIVE` |
| `bureau_balance` | 1 mois × crédit | via `SK_ID_BUREAU` → client | taux / mois en retard, historique mensuel |
| `previous_application` | 1 ancienne demande | `SK_ID_CURR` | montants, durées, parts approuvé / refusé / annulé |
| `POS_CASH_balance` | 1 mois × crédit | `SK_ID_CURR` | DPD, échéances futures |
| `credit_card_balance` | 1 mois × carte | `SK_ID_CURR` | solde, plafond, utilisation, DPD, paiements |
| `installments_payments` | 1 échéance | `SK_ID_CURR` | écarts de paiement, retards (`LATE_RATE`), sommes payées |


### 3.1 Bureau et bureau_balance


In [ ]:
def agg_bureau(bureau: pd.DataFrame) -> pd.DataFrame:
    """Retourne 1 ligne par SK_ID_CURR."""
    # TODO : groupby("SK_ID_CURR").agg(...)
    # TODO : préfixer les colonnes (ex. BUREAU_)
    raise NotImplementedError

# bureau_agg = agg_bureau(bureau)
# bureau_agg.shape

In [ ]:
# Optionnel / plus lourd
def agg_bureau_balance(bureau_balance: pd.DataFrame, bureau: pd.DataFrame) -> pd.DataFrame:
    """Agréger mois → crédit → client (stratégie à définir)."""
    # TODO
    raise NotImplementedError

### 3.2 Previous application

In [ ]:
def agg_previous(previous: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

### 3.3 POS_CASH / credit_card / installments

Même schéma : fonctions d’agrégation → 1 ligne / `SK_ID_CURR`, puis merge sur la table application.


In [ ]:
def agg_pos_cash(pos: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def agg_credit_card(cc: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def agg_installments(inst: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## 4. Merge vers une table client

Après chaque merge, la table train reste à ~307k lignes (1 demande = 1 ligne). Les NA sur colonnes agrégées correspondent surtout aux clients sans historique dans la table source.


In [ ]:
def merge_features(base: pd.DataFrame, *aggs: pd.DataFrame) -> pd.DataFrame:
    out = base.copy()
    n0 = len(out)
    for a in aggs:
        # TODO : left merge on SK_ID_CURR
        # TODO : assert len(out) == n0
        pass
    return out

# train_full = merge_features(train, bureau_agg, ...)
# test_full = merge_features(test, ...)

## 5. Sauvegarde intermédiaire

`data/interim/train_full.parquet` et `test_full.parquet` : snapshot FE complet avant split, pour rejeu / debug sans recalculer toutes les agrégations.


In [ ]:
# TODO : train_full.to_parquet(INTERIM / "train_full.parquet")
# TODO : test_full.to_parquet(INTERIM / "test_full.parquet")

## 6. Split stratifié

Dans `src/build_dataset.py` :
1. Séparation train / valid **80 / 20**, `stratify=TARGET`, `random_state=42` ;
2. écriture de `data/processed/{train,valid,test}.parquet` + `meta.json` ;
3. le test Kaggle n’a pas de `TARGET` ; aucune statistique n’y est fit.

Le déséquilibre de classes est géré à la modélisation via `class_weight="balanced"` (pas de SMOTE dans le pipeline retenu).


In [ ]:
# TODO : séparer features / TARGET
# TODO : split stratifié
# TODO : colonnes numériques vs catégorielles
# TODO : ColumnTransformer / Pipeline sklearn
# TODO : sauvegarder X_train, X_valid, y_train, y_valid dans data/processed/

raise NotImplementedError("À construire après les merges")

## 7. Reproduction

```powershell
uv run python -m src.build_dataset
```

Traçabilité : `data/processed/meta.json` (version FE, tailles, seed) et journal `choix_techniques.txt`.
